# Object Weighing
This notebook contains scripts, code samples, notes, and systems designed to work as the part of the algorithm that assigns weights to signals/objects detected in the environment.

## Project updates
After doing some more research into the future of this project, I have realized that there is a chance that all the prompt parsing code I had written might not be needed at all. I will be now trying to use `sentence_transformers` to convert the prompt and all input signals (assuming it can run at a decent speed) into vectors and comparing them directly. If this works, it would be amazing but it might require a lot of tuning and playing around with it.

## Using Sentence Transformers
The python library, `sentence_transformers` (will be referred to as `st`), is used to create text embeddings which can then be compared for similarity.

Below, is the example provided on the `st` documentation page, rewritten a little bit to be better example for this project. On top of the base example, I also added in `cosine_similarity` checks using `sklearn`.

`cosine_similarity` basically calculates the cosine of the angle between two vectors, which in this case are the embeddings of the input prompt and the signals, to calculate how similar the two embeddings are. This is better than what I was doing earlier with `sapCy` because this can very easily compare meanings without having to parse anything.

I put in a bunch of prompt types, including the full prompt and also some semi parsed variations of it. I also put in various signals including those that are very related and those that are very unrelated (these were generated using ChatGPT 5.2).

The first model that was tried was `all-MiniLM-L6-v2`. This was working pretty slow on my hardware and it also didn't *seem* accurate. Therefore, I moved to the `paraphrase-albert-small-v2` model which was significantly faster and also *seemed* more accurate.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("paraphrase-albert-small-v2")

prompt = [
    "10 year old boy in red winter clothing lost in the dense forest",
    "red winter clothing",
    "10 year old boy",
    "lost"
]  

signals = [
    "red fabric", 
    "small footprints",
    "red jacket",
    "human shape",
    "boot prints",
    "bush",
    "animal tracks",
    "red object",
]


promptEmbeddings = model.encode(prompt)
signalEmbeddings = model.encode(signals)

similarities = model.similarity(promptEmbeddings, signalEmbeddings)
similarity_score = cosine_similarity(promptEmbeddings, signalEmbeddings)

scores = {}

for i, s in enumerate(similarity_score[0]):
    scores[signals[i]] = s

scores = dict(sorted(scores.items(), key=lambda item: item[1], reverse=True))

print(scores)

### Results
Looking at the results of this basic scoring model, I am realizing that this basic "algorithm" will not cut it. After doing some more research about how similar algorithms work and the issues with `cosine_similarity` [Paper Link](https://www.aimodels.fyi/papers/arxiv/semantics-at-angle-when-cosine-similarity-works), I decided to split up the prompt using the parsing algorithm I had already written, use similarity on each part instead of on the whole prompt, and test that out. So the code for this iteration is below.

In [61]:
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer("paraphrase-albert-small-v2")

results = {
    "object": "",
    "state": [],
    "descriptions": []
}

def findObject(promptDoc):
    obj = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence that isn't the object we want
            for tok in token.children:
                if(tok.pos_ == "NOUN" and tok.dep_ in ["nsubj", "nsubjpass", "dobj"]):
                    obj = tok.text
        elif(token.dep_ == "ROOT"):
            # We have found the noun root of the sentence
            obj = token.text
    results["object"] = obj

def findState(promptDoc):
    state = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence and it is the state defining verb
            state = token.text
    if (state == ""):
        possibleState = []
        for token in promptDoc:
            if(token.pos_ == "VERB"):
                possibleState.append(token.text)
        if(len(possibleState) == 1):
            state = possibleState[0]
    if(state == ""):
        for token in promptDoc:
            if(token.pos_ == "VERB" and token.dep_ in ["amod", "acl"]):
                state = token.text
    if (state == ""):
        for token in promptDoc:
            if(token.pos_ == "ADP"):
                state = token.text
    results["state"].append(state)

def findDesc(promptDoc):
    desc = []
    for token in promptDoc:
        curr = []
        if((token.dep_ in ["amod", "nummod"] or (token.dep_ == "compound")) and len(list(token.children)) == 0):
            curr.append(token.text)
            if(token.head.pos_ == "NOUN" and token.text != results["object"]):
                curr.append(token.head.text)
                if(token.head.dep_ in ["compound", "npadvmod"] and token.head.head.pos_ in ["NOUN", "ADJ"] and token.head.head.text != results["object"]):
                    curr.append(token.head.head.text)
        if(curr != []): 
            desc.append(" ".join(desc for desc in curr if desc != results["object"]))
    results["descriptions"] = desc

def parse(prompt):
    results["object"] = ""
    results["state"] = []
    results["descriptions"] = []
    print(prompt)
    promptDoc = nlp(prompt)
    findObject(promptDoc)
    findState(promptDoc)
    findDesc(promptDoc)

prompt = [
    "10 year old boy in red winter clothing lost in the dense forest",
]  

signals = [
    "red fabric", 
    "small footprints",
    "red jacket",
    "human shape",
    "boot prints",
    "bush",
    "animal tracks",
    "red object",
]


promptEmbeddings = model.encode(prompt)
signalEmbeddings = model.encode(signals)

similarities = model.similarity(promptEmbeddings, signalEmbeddings)
similarity_score = cosine_similarity(promptEmbeddings, signalEmbeddings)

scores = {}

for i, s in enumerate(similarity_score[0]):
    scores[signals[i]] = s

scores = dict(sorted(scores.items(), key=lambda item: item[1], reverse=True))

'''
Original Custom Weights

- Object x4
- State x3
- Description x2
- Other x1
'''

objectWeight = 3.5
stateWeight = 3
descWeight = 3

parse(prompt[0])
print(results)
objectSimilarities = {}


for i, s in enumerate(cosine_similarity(model.encode([results["object"]]), model.encode(signals))[0]):
    objectSimilarities[signals[i]] = round(float(s) * 100 * 0.25 * objectWeight, 2)

objectSimilarities = dict(sorted(objectSimilarities.items(), key=lambda item: item[1], reverse=True))

stateSimilarities = {}

for i, s in enumerate(cosine_similarity(model.encode(results["state"]), model.encode(signals))[0]):
    stateSimilarities[signals[i]] = round(float(s) * 100 * 0.25 * stateWeight, 2)

stateSimilarities = dict(sorted(stateSimilarities.items(), key=lambda item: item[1], reverse=True))

descSimilarities = {}

for i, s in enumerate(cosine_similarity(model.encode(results["descriptions"]), model.encode(signals))[0]):
    descSimilarities[signals[i]] = round(float(s) * 100 * 0.25 * descWeight, 2)

descSimilarities = dict(sorted(descSimilarities.items(), key=lambda item: item[1], reverse=True))

print(objectSimilarities)
print(stateSimilarities)
print(descSimilarities)
print()

signals = set(objectSimilarities.keys()).union(stateSimilarities.keys()).union(descSimilarities.keys())
signalList = {k:max(objectSimilarities.get(k, float('-inf')), stateSimilarities.get(k, float('-inf')), descSimilarities.get(k, float('-inf'))) for k in signals}
signalList = dict(sorted(signalList.items(), key=lambda item: item[1], reverse=True))
print(signalList)

10 year old boy in red winter clothing lost in the dense forest
{'object': 'boy', 'state': ['lost'], 'descriptions': ['10 year old', 'red winter clothing', 'dense forest']}
{'human shape': 24.88, 'boot prints': 20.68, 'bush': 18.69, 'red jacket': 13.55, 'red object': 11.97, 'animal tracks': 11.06, 'red fabric': 7.36, 'small footprints': 1.86}
{'red jacket': 20.19, 'animal tracks': 17.79, 'red object': 17.43, 'human shape': 17.14, 'small footprints': 15.54, 'red fabric': 12.05, 'boot prints': 8.29, 'bush': 7.42}
{'red jacket': 16.3, 'human shape': 14.38, 'small footprints': 12.93, 'red fabric': 11.53, 'red object': 9.04, 'animal tracks': 7.91, 'bush': 6.8, 'boot prints': -3.1}

{'human shape': 24.88, 'boot prints': 20.68, 'red jacket': 20.19, 'bush': 18.69, 'animal tracks': 17.79, 'red object': 17.43, 'small footprints': 15.54, 'red fabric': 12.05}


## Implementing Open-Set Semantic Classification
While the results from the previous testing seem almost perfect, somethings aren't where I would hope they would be. For example, bush is weighted higher than small footprints which just doesn't make sense. Even after playing around with the category weights, I wasn't able to extract a output that worked for me.

What I currently have is a limited (object, state, description) closed set classifier system. My biggest hurdle is the fact that the categories--object, state, and description--aren't very narrow and therefore, the embeddings aren't unique enough to properly differentiate between actually useful signals v.s. useless ones. Therefore, I want to work on adding more categories such as "entity", "appearance", "environment", and "state" which I feel would better represent each possible chunk of the prompt and can help explain the important signals that are seen. 

This is, at least at first glance, significantly harder to do using `spaCy` so I have chosen to implement a dual-system classification algorithm. Using `spaCy`, I will first break up the prompt into meaningful chunks of information like I already do. I will also give some examples for each category that can be used as a semantic anchor. Using embeddings and `cosine_similarity`, each prompt chunk will then be classified into one of the categories based on semantic meaning rather than grammar rules.

I believe this method should lead to better results since the prompt is broken into more specific classifiers. The first implementation is shown below

In [59]:
import spacy
import math
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer("paraphrase-albert-small-v2")

results = {
    "object": "",
    "state": [],
    "descriptions": [],
    "chunks": []
}

def findObject(promptDoc):
    obj = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence that isn't the object we want
            for tok in token.children:
                if(tok.pos_ == "NOUN" and tok.dep_ in ["nsubj", "nsubjpass", "dobj"]):
                    obj = tok.text
        elif(token.dep_ == "ROOT"):
            # We have found the noun root of the sentence
            obj = token.text
    results["object"] = obj
    results["chunks"].append(obj)

def findState(promptDoc):
    state = ""
    for token in promptDoc:
        if(token.dep_ == "ROOT" and token.pos_ == "VERB"):
            # We have found the root of the sentence and it is the state defining verb
            state = token.text
    if (state == ""):
        possibleState = []
        for token in promptDoc:
            if(token.pos_ == "VERB"):
                possibleState.append(token.text)
        if(len(possibleState) == 1):
            state = possibleState[0]
    if(state == ""):
        for token in promptDoc:
            if(token.pos_ == "VERB" and token.dep_ in ["amod", "acl"]):
                state = token.text
    if (state == ""):
        for token in promptDoc:
            if(token.pos_ == "ADP"):
                state = token.text
    results["state"].append(state)
    results["chunks"].append(state)

def findDesc(promptDoc):
    desc = []
    for token in promptDoc:
        curr = []
        if((token.dep_ in ["amod", "nummod"] or (token.dep_ == "compound")) and len(list(token.children)) == 0):
            curr.append(token.text)
            if(token.head.pos_ == "NOUN" and token.text != results["object"]):
                curr.append(token.head.text)
                if(token.head.dep_ in ["compound", "npadvmod"] and token.head.head.pos_ in ["NOUN", "ADJ"] and token.head.head.text != results["object"]):
                    curr.append(token.head.head.text)
        if(curr != []): 
            desc.append(" ".join(desc for desc in curr if desc != results["object"]))
    results["descriptions"] = desc
    results["chunks"].extend(desc)

def parse(prompt):
    results["object"] = ""
    results["state"] = []
    results["descriptions"] = []
    print(prompt)
    promptDoc = nlp(prompt)
    findObject(promptDoc)
    findState(promptDoc)
    findDesc(promptDoc)

categoryExamples = {
    "entity": ["people", "person", "boy", "man", "girl", "woman", "human", "silhouette","animals", "robots", "vehicles", "drones", "objects", "people humans individuals persons",
        "children adults men women",
        "animals wildlife pets livestock",
        "robots drones autonomous machines",
        "vehicles cars trucks rovers aircraft",
        "objects items tools equipment devices",
        "missing person or target object",
        "physical thing being searched for"],
    "evidence": [
        "tracks footprints impressions",
        "animal tracks paw prints",
        "human footprints shoe prints",
        "signs of movement or presence",
        "traces left behind by an entity",
        "indirect indicators of a target",
        "marks on ground or snow"
    ],

    "appearance": ["clothing", "color", "size", "shape", "texture",  "clothing worn by a person",
        "colors and color patterns",
        "jackets coats uniforms armor",
        "size shape visual characteristics",
        "materials fabric metal plastic",
        "gear backpacks helmets equipment",
        "visible markings or features",
        "what something looks like"],
 "environment": [
        "terrain landscape surroundings",
        "forest woods jungle trees",
        "mountains valleys ravines",
        "urban areas buildings streets",
        "weather snow rain fog",
        "physical location or area"
    ],
    "state": ["lost", "damaged", "injured", "inactive", "broken", "lost missing or unaccounted for",
        "damaged broken malfunctioning",
        "injured hurt wounded",
        "inactive immobile stuck trapped",
        "operational active running",
        "powered on or powered off",
        "safe stable normal condition",
        "current condition or status"],
    "other": ["abstract", "concepts", "unrelated", "nonsense", "unknown", "generic", "abstract concepts unrelated",
        "unknown or irrelevant information",
        "nonspecific descriptive language",
        "background or filler text",
        "information not tied to categories"]
}

entityEmbeddings = [np.mean(model.encode(categoryExamples["entity"]), axis=0)]
evidenceEmbeddings = [np.mean(model.encode(categoryExamples["evidence"]), axis=0)]
appearanceEmbeddings = [np.mean(model.encode(categoryExamples["appearance"]), axis=0)]
environmentEmbeddings = [np.mean(model.encode(categoryExamples["environment"]), axis=0)]
stateEmbeddings = [np.mean(model.encode(categoryExamples["state"]), axis=0)]
otherEmbeddings = [np.mean(model.encode(categoryExamples["other"]), axis=0)]
categories = {
    "entities": [],
    "evidence": [],
    "appearance": [],
    "environment": [],
    "state": [],
    "other": []
}

categoryWeights = {
    "entities": 5,
    "appearance": 0.65,
    "state": 0.50,
    "environment": 0.3,
    "evidence": 0.55,
    "other": 0.1
}

prompt = "10 year old boy in red winter clothing lost in the dense forest"
# prompt = "six wheeled exploration rover with a damaged antenna last seen near a rocky crater on mars"
parse(prompt)
# print(results)

for chunk in results["chunks"]:
    chunkEmbeddings = model.encode([chunk])
    highestScore = max(cosine_similarity(chunkEmbeddings, entityEmbeddings)[0])
    h = "entity"

    evidenceScore = max(cosine_similarity(chunkEmbeddings, evidenceEmbeddings)[0])
    if evidenceScore > highestScore:
        highestScore = evidenceScore
        h = "evidence"

    appScore = max(cosine_similarity(chunkEmbeddings, appearanceEmbeddings)[0])
    if appScore > highestScore:
        highestScore = appScore
        h = "app"

    envScore = max(cosine_similarity(chunkEmbeddings, environmentEmbeddings)[0])
    if envScore > highestScore:
        highestScore = envScore
        h = "env"

    stateScore = max(cosine_similarity(chunkEmbeddings, stateEmbeddings)[0])
    if stateScore > highestScore:
        highestScore = stateScore
        h = "state"

    otherScore = max(cosine_similarity(chunkEmbeddings, otherEmbeddings)[0])
    if otherScore > highestScore:
        highestScore = otherScore
        h = "other"

    # print(f"{chunk} {highestScore * 100:.2f}% ({h})")

    match h:
        case 'entity':
            categories["entities"].append(chunk)
        case 'evidence':
            categories["evidence"].append(chunk)
        case 'app':
            categories["appearance"].append(chunk)
        case 'env':
            categories["environment"].append(chunk)
        case 'state':
            categories["state"].append(chunk)
        case 'other':
            categories["other"].append(chunk)

print(categories)

signals = [
    "red fabric", 
    "small footprints",
    "red jacket",
    "human shape",
    "boot prints",
    "bush",
    "animal tracks",
    "red object",
]

# signals = [
#     "metal tracks in reddish soil",
#     "radio signal intermittently lost",
#     "wheel impressions with uneven spacing",
#     "reflective solar panel fragment",
#     "rocky terrain",
#     "dust storm",
#     "small metallic object",
#     "antenna-shaped debris"
# ]

promptEmbedding = model.encode([prompt])
signalEmbeddings = model.encode(signals)
similarities = {}

def curve(x, k = 3.2, l = 0.05):
    # return 1
    if x <= l:
        return x / l
    else:
        return math.exp(k * (x - l))

for key in categories.keys():
    similarities[key] = {}
    if len(categories[key]) == 0:
        continue
    for i, s in enumerate(cosine_similarity(model.encode(categories[key]), signalEmbeddings)[0]):
        similarities[key][signals[i]] = float(s) * curve(float(cosine_similarity(promptEmbedding, model.encode([signals[i]]))[0][0])) * categoryWeights.get(key, 0.0)
    similarities[key] = dict(sorted(similarities[key].items(), key=lambda item: item[1], reverse=True))

print(similarities)

# print(cosine_similarity(model.encode(["10 year old boy in red winter clothing lost in the dense forest"]), model.encode(["animal tracks"])))
# signals = set(categories.values().keys()).union(stateSimilarities.values().keys()).union(descSimilarities.values().keys())
# signalList = {k:max(objectSimilarities.get(k, float('-inf')), stateSimilarities.get(k, float('-inf')), descSimilarities.get(k, float('-inf'))) for k in signals}

signalList = {}
for signal in signals:
    maxSignal = float('-inf')
    for key in categories.keys():
        s = round(similarities[key].get(signal, float('-inf')) * 100, 2)
        if s > maxSignal:
            maxSignal = s
    signalList[signal] = maxSignal
signalList = dict(sorted(signalList.items(), key=lambda item: item[1], reverse=True))
print(signalList)

10 year old boy in red winter clothing lost in the dense forest
{'entities': ['boy', '10 year old'], 'evidence': [], 'appearance': ['red winter clothing'], 'environment': ['dense forest'], 'state': ['lost'], 'other': []}
{'entities': {'red jacket': 2.3548449771230713, 'human shape': 1.6913709754285562, 'red object': 1.3310583737961705, 'animal tracks': 1.120304887755691, 'boot prints': 0.9682194849384373, 'red fabric': 0.7329731836491892, 'bush': 0.7072768838830568, 'small footprints': 0.16771300775621917}, 'evidence': {}, 'appearance': {'red jacket': 1.5966060665464539, 'red fabric': 0.8139913089918214, 'red object': 0.4908768231623441, 'human shape': 0.1392067834722401, 'animal tracks': 0.13857625933185505, 'small footprints': 0.13850386929682965, 'boot prints': 0.036797792214164016, 'bush': -0.022388057339892353}, 'environment': {'animal tracks': 0.18811901782818066, 'red jacket': 0.1418823505232664, 'small footprints': 0.13532486877676786, 'red object': 0.11714367046049312, 'red fa

### Results
Through a lot of trial and error, and guessing, and adding in new things that I thought/hoped would help, I ended up with the above algorithm. It works much better than previously. At least, a little bit. Maybe. It seems very subjective at the moment to be honest, but what I've realized is that I'm actually testing signals against the prompt for similarity, rather than usefulness. For example, indirect evidence like "small footprints" isn't scored very high because nothing like it is mentioned in the prompt. It confuses being expected with being important.

To fix this, I will have to re-structure the architecture of my algorithm. I need more similarity metrics for each category because they each serve a different purpose. For example, appearances describe the target, environment sets the context, and evidence supports the target's state. Because evidence is inferential, scoring it the same way as everything else doesn't work. THe prompt has no clue about the evidence so it will never score as high as the other categories, while possibly being more important.

Therefore, the prompt must define a target hypothesis, outlining what, how, where, etc. about the target, and then the signals are evaluated based on how much they support the hypothesis rather than how similar they are to the prompt. For example, if the signal is small footprints, rather than determining how similar it is to the prompt, I need to determine how likely is it to support the hypothesis. Which, now that I think about it, is basically what my starting goal was. Find signals that are likely to lead to the target based on what you know about the target and what you can see from the current environment. Footprints have a high probability of leading you to another clue/the target but obviously, the prompt wouldn't know about them.